## Using The Movie Database API to curate a Movie Dataset
This notebook is used to curate a dataset of movies, specifically Hallmark movies, from TMDB using their free API.


### The Obvious First Attempt
At first, this seemed like a fairly straightforward task. Filter for movies from Hallmark using the discover feature with the with_companies parameter set to Hallmark Media. Easy, right?
Well, I did try that. Turns out I was only getting Hallmark movies produced directly by Hallmark Media. Hallmark partners with several other production companies, including very small production companies apparently created for specific Hallmark series. Who knew?

### The Scavenger Hunt
That led me to trying to find existing sources of movies and seeing if I could create a larger list of production companies that produce Hallmark movies. I checked existing blogs (shoutout to the Hallmark superfans keeping meticulous spreadsheets), IMDB lists, and cross-referenced everything against TMDB and HallmarkChannel.com listings to build a nearly complete list of production companies.
### Filling the Gaps
After pulling all the data, I noticed there were still some gaps. For some movies, the production company data was just...missing from TMDB. Not wanting to abandon these orphaned films, I needed a way to add in new movies and missing production companies by cross-referencing my IMDB list and spot-checking new releases from HallmarkChannel.com.

## Prerequisites

### Obtain API key from The Movie Database 
https://developer.themoviedb.org/docs/getting-started

### Jupyter Notebook
https://jupyter.org/install

### Set up environment to store API keys
#### Step 1. In terminal/command prompt
touch .env  # Mac/Linux

type nul > .env  # Windows


#### **Step 2: Add your API key to `.env`**

Open `.env` and add:

TMDB_API_KEY=your_actual_api_key_here

In [1]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import time
import json
load_dotenv()
api_key = os.getenv('TMDB_API_KEY')

In [ ]:
company_ids = '1311|1370|2090|3201|3282|3431|3958|183449|202962|4056|5056|6345|9027|159010|10102|10967|12799|13404|17156|17302|20068|25954|27395|52429|53015|64216|70188|77762|83228|111905|115965|119056|124099|147078|147079|147926|164530|166637|184772|192262|254320|254322|254323|5225|6438|6679|7119|14987|48782|84293|124756|162068|184816|219548|6528|33|2865|194610|7188|3995|15645'

url = f'https://api.themoviedb.org/3/discover/movie?api_key={api_key}&with_companies={company_ids}&primary_release_date.gte=2010-01-01&sort_by=primary_release_date.desc'

# Collect all movies
all_movies = []
page = 1

while True:
    response = requests.get(f"{url}&page={page}")
    data = response.json()
    
    if not data['results']:
        break
    
    all_movies.extend(data['results'])
    
    if page >= data['total_pages']:
        break
    
    page += 1

print(f"Collected {len(all_movies)} movies. Now fetching detailed information...")

# Get full details
movie_data = []

for i, m in enumerate(all_movies):
    if i % 10 == 0:
        print(f"Processing movie {i+1}/{len(all_movies)}")
    
    movie_id = m.get('id')
    movie_url = f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}&append_to_response=keywords'
    response = requests.get(movie_url)
    
    if response.status_code == 200:
        details = response.json()
        
        # Check homepage
        homepage = details.get('homepage', '').lower()   
        
        # Extract clean parsed data
        prod_companies = ', '.join([c['name'] for c in details.get('production_companies', [])])
        genres = ', '.join([g['name'] for g in details.get('genres', [])])
        keywords_data = details.get('keywords', {}).get('keywords', [])
        keywords = ', '.join([k['name'] for k in keywords_data])
        
                # Determine Hallmark status
        if 'hallmark' in homepage:
            hallmark_status = 'Confirmed'
        elif "Hallmark" in prod_companies:
            hallmark_status = 'Confirmed'
        elif homepage == '' or homepage == 'n/a':
            hallmark_status = 'Unknown'
        else:
            hallmark_status = 'Non-Hallmark'
        
        movie_data.append({
            'Title': details.get('title', m['title']),
            'Year': m['release_date'][:4] if m.get('release_date') else 'N/A',
            'Release Date': m.get('release_date', 'N/A'),
            'Hallmark_Status': hallmark_status,
            'Production Companies': prod_companies,
            'Genres': genres,
            'Keywords': keywords,
            'Runtime': details.get('runtime', 'N/A'),
            'Budget': details.get('budget', 0),
            'Revenue': details.get('revenue', 0),
            'Popularity': m.get('popularity', 'N/A'),
            'Rating': m.get('vote_average', 'N/A'),
            'Votes': m.get('vote_count', 0),
            'Plot': m.get('overview', 'No plot available'),
            'IMDb_ID': details.get('imdb_id', 'N/A'),
            'Homepage': details.get('homepage', 'N/A'),
            'poster_path': m.get('poster_path', 'N/A'),
            'id': movie_id,
            'raw_data': json.dumps(details)
        })
    
    time.sleep(0.25)

df = pd.DataFrame(movie_data)

print(f"\nTotal movies: {len(df)}")
print(f"Confirmed Hallmark: {len(df[df['Hallmark_Status'] == 'Confirmed'])}")
print(f"Unknown (no homepage): {len(df[df['Hallmark_Status'] == 'Unknown'])}")
print(f"Non-Hallmark: {len(df[df['Hallmark_Status'] == 'Non-Hallmark'])}")

# Save all data with status
df.to_csv('hallmark_movies_with_status_2.csv', index=False) 
print("\nSaved to 'hallmark_movies_with_status_2.csv'")

In [ ]:
## Missing production companies 

# 1. UPDATED: Your new list of company IDs
company_ids = '159010|202962|183449|36152|197707|137887|12532|263912|53015|152354|194610|147599'  # Replace with your updated list

# 2. NEW: Manual list of Hallmark movie titles (no production company listed)
# Add movie titles that I know are Hallmark but don't have production company data from TMBD
MANUAL_HALLMARK_TITLES = [
    "Love of the Irish",
"My Argentine Heart",
"The Perfect Setting",
"Polar Opposites",
"The Wish Swap",
"Sisterhood, Inc.",
"The Royal We",
"An Unexpected Valentine",
"Reality Bites: A Hannah Swensen Mystery",
"Return to Office",
"Royal-ish",
"The Reluctant Royal",
"Hearts Around the Table: Jenna's First Love",
"Hearts Around the Table: Shari's Second Act",
"Mystery Island: Winner Takes All",
"Cooked to Death: A Hannah Swensen Mystery",
"Hearts Around the Table: Josh's Third Serving",
"Hearts Around the Table: Kiki's Fourth Ingredient",
"Journey to You",
"The Love Club Moms: Tory",
"Signed, Sealed, Delivered: To the Moon and Back",
"Love in the Clouds",
"Hats Off to Love",
"Love on the Danube: Love Song",
"Love on the Danube: Royal Getaway",
"Mystery Island: Play for Keeps",
"Mystery Island: House Rules",
"A Machu Picchu Proposal",
"A Newport Christmas",
"To Barcelona, with Love",
"To Barcelona, Forever",
"Pie to Die For: A Hannah Swensen Mystery",
"True Justice 2: Eye for an Eye",
"A Royal Montana Christmas",
"Providence Falls",
"Thief of Fate",
"Double Scoop",
"Catch of the Day",
"Home Turf",
"Adventures in Love & Birding",
"Haul Out the Halloween",
"The Love Club Moms",
"A Make or Break Holiday",
"A Suite Holiday Romance",
"The Christmas Cup",
"The Snow Must Go On",
"The Christmas Baby",
"Oy to the World!",
"We Met in December",
"Christmas at the Catnip Cafe",
"A Keller Christmas Vacation",
"Holiday Touchdown: A Bills Love Story",
"The Groomsmen Last Dance",
"A Christmas Angel Match",
"An Alpine Holiday",
"Three Wisest Men",
"Christmas Above the Clouds",
"A Grand Ole Opry Christmas"
]

# Step 1: Discover movies by production company
url = f'https://api.themoviedb.org/3/discover/movie?api_key={api_key}&with_companies={company_ids}&primary_release_date.gte=2010-01-01&sort_by=primary_release_date.desc'

# Collect all movies from discovery
all_movies = []
page = 1

print("Fetching movies from production companies...")
while True:
    response = requests.get(f"{url}&page={page}")
    
    if response.status_code != 200:
        print(f"Error on page {page}: {response.status_code}")
        break
    
    data = response.json()
    
    if not data.get('results'):
        break
    
    all_movies.extend(data['results'])
    print(f"Fetched page {page}: {len(data['results'])} movies (Total: {len(all_movies)})")
    
    if page >= data.get('total_pages', 1):
        break
    
    page += 1
    time.sleep(0.25)  # Rate limiting

print(f"\nCollected {len(all_movies)} movies from discovery.")

# Step 2: Search for manual Hallmark titles
print(f"\nSearching for {len(MANUAL_HALLMARK_TITLES)} manually specified Hallmark movies...")

for title in MANUAL_HALLMARK_TITLES:
    print(f"Searching for: {title}")
    search_url = f'https://api.themoviedb.org/3/search/movie'
    params = {'query': title, 'api_key': api_key}
    response = requests.get(search_url, params=params)
    
    if response.status_code == 200:
        results = response.json().get('results', [])
        if results:
            # Get the first result (most relevant)
            movie = results[0]
            
            # Check if already in list (avoid duplicates)
            if not any(m['id'] == movie['id'] for m in all_movies):
                all_movies.append(movie)
                print(f"  ✓ Added: {movie['title']} ({movie.get('release_date', 'N/A')[:4]})")
            else:
                print(f"  → Already in list")
        else:
            print(f"  ✗ Not found")
    
    time.sleep(0.25)

print(f"\nTotal movies to process: {len(all_movies)}")

# Step 3: Get full details for all movies
print("\nFetching detailed information...")
movie_data = []

for i, m in enumerate(all_movies):
    if i % 10 == 0:
        print(f"Processing movie {i+1}/{len(all_movies)}")
    
    movie_id = m.get('id')
    movie_url = f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}&append_to_response=keywords'
    response = requests.get(movie_url)
    
    if response.status_code == 200:
        details = response.json()
        
        # Get movie title
        title = details.get('title', m.get('title', ''))
        
        # Check homepage
        homepage = details.get('homepage', '').lower()
        
        # Extract production companies
        prod_companies_list = details.get('production_companies', [])
        prod_companies = ', '.join([c['name'] for c in prod_companies_list])
        
        # Extract other data
        genres = ', '.join([g['name'] for g in details.get('genres', [])])
        keywords_data = details.get('keywords', {}).get('keywords', [])
        keywords = ', '.join([k['name'] for k in keywords_data])
        
        # Determine Hallmark status with improved logic
        if 'hallmark' in homepage:
            hallmark_status = 'Confirmed'
        elif "Hallmark" in prod_companies:
            hallmark_status = 'Confirmed'
        elif title in MANUAL_HALLMARK_TITLES:
            hallmark_status = 'Confirmed (Manual)'
        elif not prod_companies or homepage == '' or homepage == 'n/a':
            hallmark_status = 'Unknown'
        else:
            hallmark_status = 'Non-Hallmark'
        
        movie_data.append({
            'Title': title,
            'Year': m['release_date'][:4] if m.get('release_date') else 'N/A',
            'Release Date': m.get('release_date', 'N/A'),
            'Hallmark_Status': hallmark_status,
            'Production Companies': prod_companies,
            'Genres': genres,
            'Keywords': keywords,
            'Runtime': details.get('runtime', 'N/A'),
            'Budget': details.get('budget', 0),
            'Revenue': details.get('revenue', 0),
            'Popularity': m.get('popularity', 'N/A'),
            'Rating': m.get('vote_average', 'N/A'),
            'Votes': m.get('vote_count', 0),
            'Plot': m.get('overview', 'No plot available'),
            'IMDb_ID': details.get('imdb_id', 'N/A'),
            'Homepage': details.get('homepage', 'N/A'),
            'poster_path': m.get('poster_path', 'N/A'),
            'id': movie_id,
            'raw_data': json.dumps(details)
        })
    else:
        print(f"  Error fetching details for movie ID {movie_id}: {response.status_code}")
    
    time.sleep(0.25)

# Create DataFrame
df = pd.DataFrame(movie_data)

# Summary statistics
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Total movies: {len(df)}")
print(f"Confirmed Hallmark: {len(df[df['Hallmark_Status'] == 'Confirmed'])}")
print(f"Confirmed Hallmark (Manual): {len(df[df['Hallmark_Status'] == 'Confirmed (Manual)'])}")
print(f"Unknown (no homepage/companies): {len(df[df['Hallmark_Status'] == 'Unknown'])}")
print(f"Non-Hallmark: {len(df[df['Hallmark_Status'] == 'Non-Hallmark'])}")

# Save to CSV
output_file = 'hallmark_movies_complete.csv'
df.to_csv(output_file, index=False)
print(f"\nData saved to: {output_file}")

# Display sample
print(f"\nSample of manually added Hallmark movies:")
manual_movies = df[df['Hallmark_Status'] == 'Confirmed (Manual)']
if not manual_movies.empty:
    print(manual_movies[['Title', 'Year', 'Production Companies']].head(10))




In [17]:
df.head()

,Title,Year,Release Date,Hallmark_Status,Production Companies,Genres,Keywords,Runtime,Budget,Revenue,Popularity,Rating,Votes,Plot,IMDb_ID,Homepage,poster_path,id,raw_data
0,A Make or Break Holiday,2025.0,2025-12-20,Confirmed,Hallmark Media,Romance,NaN,84,0,0,1.5217,0.0,0,"Despite recently going on a break, couple Liv ...",tt38354998,NaN,/i0CYe2VUX6TCA488Br5oO4yiWKT.jpg,1547927,"{""adult"": false, ""backdrop_path"": null, ""belon..."
1,She's Making a List,2025.0,2025-12-06,Confirmed,Hallmark Media,"Romance, Comedy, TV Movie","christmas romance, holiday romance",84,0,0,2.7428,6.0,1,Naughty or Nice inspector Isabel Haynes doesn’...,tt38354965,https://www.hallmarkchannel.com/shes-making-a-...,/mLxAvf63Xhw0E6ymh86n3pQp2PI.jpg,1516208,"{""adult"": false, ""backdrop_path"": ""/eATVM26gT0..."
2,Christmas at the Catnip Cafe,2025.0,2025-11-30,Confirmed,"Hallmark Media, Lighthouse Pictures","Romance, TV Movie, Comedy",holiday romance,84,0,0,3.1942,6.5,3,When a marketing executive learns she’s short ...,tt37985000,https://www.hallmarkchannel.com/christmas-at-t...,/4OdcIegFxwFIflJ0N0hXUXNtV93.jpg,1545999,"{""adult"": false, ""backdrop_path"": ""/h5KJbGlz1d..."
3,A Grand Ole Opry Christmas,2025.0,2025-11-29,Confirmed,Hallmark Media,"Drama, Romance, TV Movie",NaN,90,0,0,8.4331,6.0,1,"Gentry Woods, who gave up music after her fath...",tt37891014,https://www.hallmarkchannel.com/a-grand-ole-op...,/y5NqsOFBum7LyX6QHIakwG1plnI.jpg,1535221,"{""adult"": false, ""backdrop_path"": ""/uG7iSiw2RY..."
4,The Snow Must Go On,2025.0,2025-11-28,Confirmed,"Hallmark Media, Inferno Pictures","Romance, TV Movie, Comedy, Drama",christmas,84,0,0,1.7006,1.0,1,"Nearly a decade after appearing on Broadway, I...",tt38354943,https://www.hallmarkchannel.com/the-snow-must-...,/zixxwy8GbxnqbKdViggwnoR611R.jpg,1547920,"{""adult"": false, ""backdrop_path"": ""/zkwVUlX97E..."


In [18]:
## Adding a Filter if one of the production companies was Hallmark vs. a different production company 

def Hallmark_Produced(prod):
    # Handle NaN or non-string values
    if not isinstance(prod, str):
        return "Unknown"
    
    production_companies = prod.split(",")
    
    for company in production_companies:
        if "Hallmark" in company:
            return "Hallmark Production"
    
    return "Other Production Company"
df['Hallmark_Production'] = df.apply(lambda row: Hallmark_Produced(row['Production Companies']), axis =1)

In [ ]:
## Pulling details about the cast for each movies

def add_credits (movie_id):
    response = requests.get(f'https://api.themoviedb.org/3/movie/{movie_id}/credits?api_key={api_key}')
    data = response.json()
    return data
df['credits'] = df.apply(lambda row: add_credits(row['id']), axis =1)  

In [ ]:
#Extracting out the cast per movies

def get_cast(credits):
    try:
        cast = credits.get('cast', [])
        return cast
    except:
        return None
df['cast'] = df.apply(lambda row: get_cast(row['credits']), axis = 1)

In [ ]:
def actor_names(cast):
    actors_list = []
        # Extract actor names
    for actor in cast:
        actors_list.append(actor['name'])
    return actors_list

df['actor_names'] = df.apply(lambda row: actor_names(row['cast']), axis =1)

In [16]:
## I already ran those blocks of code above and saved the data. 

df1 = pd.read_csv('/hallmark_movies_complete.csv')

df2 = pd.read_csv('/hallmark_movies_with_status_2.csv')

# Combine dataframes
df = pd.concat([df1, df2], ignore_index=True)

print(f"Before dedup: {len(df)} movies")

# Remove duplicates by Title
df = df.drop_duplicates(subset=['Title'], keep='first')

print(f"After dedup: {len(df)} movies")



Before dedup: 4048 movies
After dedup: 3588 movies


In [50]:

## Save file
#df.to_csv('Hallmark_Movies_2010_2025.csv')

## Reading in previously saved dataset
df = pd.read_csv('/Hallmark_Movies_2010_2025_2.csv')
df.shape

(3588, 24)

In [51]:
# double checking if previously missed titles are present
df[df['Title']=="She's Making a List"] 

,Unnamed: 0,Title,Year,Release Date,Hallmark_Status,Production Companies,Genres,Keywords,Runtime,Budget,...,Plot,IMDb_ID,Homepage,poster_path,id,raw_data,Hallmark_Production,credits,cast,actor_names
1,1,She's Making a List,2025.0,2025-12-06,Confirmed,Hallmark Media,"Romance, Comedy, TV Movie","christmas romance, holiday romance",84,0,...,Naughty or Nice inspector Isabel Haynes doesn’...,tt38354965,https://www.hallmarkchannel.com/shes-making-a-...,/mLxAvf63Xhw0E6ymh86n3pQp2PI.jpg,1516208,"{""adult"": false, ""backdrop_path"": ""/eATVM26gT0...",Hallmark Production,"{'id': 1516208, 'cast': [{'adult': False, 'gen...","[{'adult': False, 'gender': 2, 'id': 134673, '...","['Andrew W. Walker', 'Lacey Chabert']"


## Joining TMDB dataset with IMDB. 
### IMDB has more users and is a reliable source for movie ratings. 
Lets join the data from IMDB available here https://datasets.imdbws.com/


In [55]:
ratings = pd.read_csv('title.ratings_12_9_2025.tsv',sep='\t')

In [56]:
ratings.head()


,tconst,averageRating,numVotes
0,tt0000001,5.7,2188
1,tt0000002,5.5,307
2,tt0000003,6.4,2274
3,tt0000004,5.1,197
4,tt0000005,6.2,3012


In [73]:
## We are doing a left because I want to preserve all of the data in my curated Hallmark dataset and add IMDB ratings information even if there is not data YET available. 
# Some new movies are not present in these datasets.

df_w_ratings = df.merge(ratings, left_on='IMDb_ID', right_on='tconst', how='left')
df_w_ratings = df_w_ratings[df_w_ratings['Year'].notna()]

In [74]:
df_w_ratings.head()

,Unnamed: 0,Title,Year,Release Date,Hallmark_Status,Production Companies,Genres,Keywords,Runtime,Budget,...,poster_path,id,raw_data,Hallmark_Production,credits,cast,actor_names,tconst,averageRating,numVotes
0,0,A Make or Break Holiday,2025.0,2025-12-20,Confirmed,Hallmark Media,Romance,NaN,84,0,...,/i0CYe2VUX6TCA488Br5oO4yiWKT.jpg,1547927,"{""adult"": false, ""backdrop_path"": null, ""belon...",Hallmark Production,"{'id': 1547927, 'cast': [{'adult': False, 'gen...","[{'adult': False, 'gender': 1, 'id': 1446320, ...","['Hunter King', 'Evan Roderick']",NaN,NaN,NaN
1,1,She's Making a List,2025.0,2025-12-06,Confirmed,Hallmark Media,"Romance, Comedy, TV Movie","christmas romance, holiday romance",84,0,...,/mLxAvf63Xhw0E6ymh86n3pQp2PI.jpg,1516208,"{""adult"": false, ""backdrop_path"": ""/eATVM26gT0...",Hallmark Production,"{'id': 1516208, 'cast': [{'adult': False, 'gen...","[{'adult': False, 'gender': 2, 'id': 134673, '...","['Andrew W. Walker', 'Lacey Chabert']",NaN,NaN,NaN
2,2,Christmas at the Catnip Cafe,2025.0,2025-11-30,Confirmed,"Hallmark Media, Lighthouse Pictures","Romance, TV Movie, Comedy",holiday romance,84,0,...,/4OdcIegFxwFIflJ0N0hXUXNtV93.jpg,1545999,"{""adult"": false, ""backdrop_path"": ""/h5KJbGlz1d...",Hallmark Production,"{'id': 1545999, 'cast': [{'adult': False, 'gen...","[{'adult': False, 'gender': 2, 'id': 62909, 'k...","['Paul Campbell', 'Erin Cahill']",tt37985000,6.9,310.0
3,3,A Grand Ole Opry Christmas,2025.0,2025-11-29,Confirmed,Hallmark Media,"Drama, Romance, TV Movie",NaN,90,0,...,/y5NqsOFBum7LyX6QHIakwG1plnI.jpg,1535221,"{""adult"": false, ""backdrop_path"": ""/uG7iSiw2RY...",Hallmark Production,"{'id': 1535221, 'cast': [{'adult': False, 'gen...","[{'adult': False, 'gender': 1, 'id': 59750, 'k...","['Nikki DeLoach', 'Kristoffer Polaha', 'Sharon...",tt37891014,7.1,383.0
4,4,The Snow Must Go On,2025.0,2025-11-28,Confirmed,"Hallmark Media, Inferno Pictures","Romance, TV Movie, Comedy, Drama",christmas,84,0,...,/zixxwy8GbxnqbKdViggwnoR611R.jpg,1547920,"{""adult"": false, ""backdrop_path"": ""/zkwVUlX97E...",Hallmark Production,"{'id': 1547920, 'cast': [{'adult': False, 'gen...","[{'adult': False, 'gender': 1, 'id': 205982, '...","['Heather Hemmens', 'Corey Cott', 'Kaelyn Yoon...",tt38354943,6.3,219.0


In [75]:
df_w_ratings.shape

(3587, 27)

In [76]:
df_w_ratings.describe()

,Unnamed: 0,Year,Runtime,Budget,Revenue,Popularity,Rating,Votes,id,averageRating,numVotes
count,3587.000000,3587.000000,3587.000000,3.587000e+03,3.587000e+03,3587.000000,3587.000000,3587.000000,3.587000e+03,3510.000000,3510.000000
mean,2026.968776,2018.891553,86.127962,3.174118e+06,1.082352e+07,3.236818,5.654078,238.830220,6.983999e+05,5.540057,9220.387179
std,1195.892020,4.201055,20.027590,2.015461e+07,8.443017e+07,4.646863,1.509936,1089.647626,3.958662e+05,1.126912,45557.673115
min,0.000000,1998.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,7.978000e+03,1.200000,5.000000
25%,992.500000,2016.000000,84.000000,0.000000e+00,0.000000e+00,2.029000,5.110000,9.000000,3.936980e+05,4.925000,331.000000
50%,2016.000000,2020.000000,87.000000,0.000000e+00,0.000000e+00,2.845100,5.906000,22.000000,6.392890e+05,5.700000,760.500000
75%,3087.500000,2022.000000,90.000000,0.000000e+00,0.000000e+00,3.877700,6.500000,52.000000,1.020538e+06,6.300000,1767.000000
max,4047.000000,2027.000000,262.000000,5.839000e+08,1.671537e+09,179.395500,10.000000,21121.000000,1.585877e+06,9.800000,961655.000000


In [77]:
# Filter for Just Movies Made by Hallmark or for Hallmark Channel
df_w_ratings.groupby('Hallmark_Status').count()

,Unnamed: 0,Title,Year,Release Date,Production Companies,Genres,Keywords,Runtime,Budget,Revenue,...,poster_path,id,raw_data,Hallmark_Production,credits,cast,actor_names,tconst,averageRating,numVotes
Hallmark_Status,,,,,,,,,,,,,,,,,,,,,
Confirmed,757,757,757,757,747,755,548,757,757,757,...,757,757,757,757,757,757,757,746,746,746
Confirmed (Manual),3,3,3,3,3,3,1,3,3,3,...,3,3,3,3,3,3,3,3,3,3
Non-Hallmark,1267,1267,1267,1267,1267,1267,774,1267,1267,1267,...,1263,1267,1267,1267,1267,1267,1267,1246,1246,1246
Unknown,1560,1560,1560,1560,1560,1551,799,1560,1560,1560,...,1545,1560,1560,1560,1560,1560,1560,1515,1515,1515


In [78]:
## Filtering my data to JUST Hallmark Movies. 
# The previous dataset included hallmark movies, and all movies produced by production companies that make movies for hallmark but are not availabe on hallmark. 

df_hallmark =df_w_ratings[(df_w_ratings['Hallmark_Status']=="Confirmed") |(df_w_ratings['Hallmark_Status']== "Confirmed (Manual)" ) ]


In [79]:
df_hallmark.shape

(760, 27)

In [80]:
df_hallmark.describe()

,Unnamed: 0,Year,Runtime,Budget,Revenue,Popularity,Rating,Votes,id,averageRating,numVotes
count,760.000000,760.000000,760.000000,7.600000e+02,760.000000,760.000000,760.000000,760.000000,7.600000e+02,749.000000,749.000000
mean,1305.880263,2019.338158,84.876316,5.644781e+04,0.006579,3.026848,6.342276,38.764474,7.528709e+05,6.429907,1591.097463
std,1265.342932,3.914113,9.883136,6.491389e+05,0.181369,1.791926,0.923519,37.186650,3.945990e+05,0.497359,1045.268512
min,0.000000,1998.000000,0.000000,0.000000e+00,0.000000,0.007100,0.000000,0.000000,4.090000e+04,4.400000,19.000000
25%,278.750000,2017.000000,84.000000,0.000000e+00,0.000000,2.220475,6.000000,17.000000,4.463405e+05,6.100000,915.000000
50%,529.500000,2020.000000,84.000000,0.000000e+00,0.000000,2.888100,6.400000,30.000000,6.533620e+05,6.400000,1390.000000
75%,2507.000000,2023.000000,85.000000,0.000000e+00,0.000000,3.590050,6.800000,49.000000,1.056734e+06,6.800000,1943.000000
max,4037.000000,2025.000000,180.000000,1.100000e+07,5.000000,28.428800,10.000000,501.000000,1.547929e+06,8.200000,12503.000000


In [81]:
df_hallmark.head()

,Unnamed: 0,Title,Year,Release Date,Hallmark_Status,Production Companies,Genres,Keywords,Runtime,Budget,...,poster_path,id,raw_data,Hallmark_Production,credits,cast,actor_names,tconst,averageRating,numVotes
0,0,A Make or Break Holiday,2025.0,2025-12-20,Confirmed,Hallmark Media,Romance,NaN,84,0,...,/i0CYe2VUX6TCA488Br5oO4yiWKT.jpg,1547927,"{""adult"": false, ""backdrop_path"": null, ""belon...",Hallmark Production,"{'id': 1547927, 'cast': [{'adult': False, 'gen...","[{'adult': False, 'gender': 1, 'id': 1446320, ...","['Hunter King', 'Evan Roderick']",NaN,NaN,NaN
1,1,She's Making a List,2025.0,2025-12-06,Confirmed,Hallmark Media,"Romance, Comedy, TV Movie","christmas romance, holiday romance",84,0,...,/mLxAvf63Xhw0E6ymh86n3pQp2PI.jpg,1516208,"{""adult"": false, ""backdrop_path"": ""/eATVM26gT0...",Hallmark Production,"{'id': 1516208, 'cast': [{'adult': False, 'gen...","[{'adult': False, 'gender': 2, 'id': 134673, '...","['Andrew W. Walker', 'Lacey Chabert']",NaN,NaN,NaN
2,2,Christmas at the Catnip Cafe,2025.0,2025-11-30,Confirmed,"Hallmark Media, Lighthouse Pictures","Romance, TV Movie, Comedy",holiday romance,84,0,...,/4OdcIegFxwFIflJ0N0hXUXNtV93.jpg,1545999,"{""adult"": false, ""backdrop_path"": ""/h5KJbGlz1d...",Hallmark Production,"{'id': 1545999, 'cast': [{'adult': False, 'gen...","[{'adult': False, 'gender': 2, 'id': 62909, 'k...","['Paul Campbell', 'Erin Cahill']",tt37985000,6.9,310.0
3,3,A Grand Ole Opry Christmas,2025.0,2025-11-29,Confirmed,Hallmark Media,"Drama, Romance, TV Movie",NaN,90,0,...,/y5NqsOFBum7LyX6QHIakwG1plnI.jpg,1535221,"{""adult"": false, ""backdrop_path"": ""/uG7iSiw2RY...",Hallmark Production,"{'id': 1535221, 'cast': [{'adult': False, 'gen...","[{'adult': False, 'gender': 1, 'id': 59750, 'k...","['Nikki DeLoach', 'Kristoffer Polaha', 'Sharon...",tt37891014,7.1,383.0
4,4,The Snow Must Go On,2025.0,2025-11-28,Confirmed,"Hallmark Media, Inferno Pictures","Romance, TV Movie, Comedy, Drama",christmas,84,0,...,/zixxwy8GbxnqbKdViggwnoR611R.jpg,1547920,"{""adult"": false, ""backdrop_path"": ""/zkwVUlX97E...",Hallmark Production,"{'id': 1547920, 'cast': [{'adult': False, 'gen...","[{'adult': False, 'gender': 1, 'id': 205982, '...","['Heather Hemmens', 'Corey Cott', 'Kaelyn Yoon...",tt38354943,6.3,219.0


In [67]:
## Saving my Hallmark Dataset. 

df_hallmark.to_csv("Hallmark_Movies_Filtered_2010_2025.csv")